# DIM_ESTABELECIMENTO

Estabelecimentos de saúde. Une o cadastro CNES com os CNES que aparecem na produção
(SIA, SIH e leitos), para que nenhuma linha de fato fique sem estabelecimento.

- **Nome:** do cadastro; ausente, vira `NAO CADASTRADO - <CNES>`.
- **Tipo de unidade e gestão:** valor mais frequente (`STATS_MODE`) na produção do SIA.
- **Município, região e macrorregião:** do cadastro, ou o município mais frequente nos atendimentos.
  Ficam desnormalizados na dimensão, como recomenda o star schema.

**Estratégia de carga:** incremental com `MERGE` (SCD tipo 1). Registros novos recebem a surrogate key da sequence; registros existentes só são atualizados se algum atributo mudou. Reexecutar a carga não duplica linhas nem troca as chaves.

**Membro desconhecido:** `SK = -1` ("NAO INFORMADO"), usado pelas fatos quando o código de origem não existe na dimensão.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import mesclar, executar, consultar

In [ ]:
MEMBRO_DESCONHECIDO = """
MERGE INTO DIM_ESTABELECIMENTO d
USING (SELECT -1 AS SK FROM DUAL) s ON (d.SK_ESTABELECIMENTO = s.SK)
WHEN NOT MATCHED THEN INSERT
    (SK_ESTABELECIMENTO, COD_CNES, NOME_ESTABELECIMENTO, TIPO_UNIDADE, TIPO_GESTAO,
     NOME_MUNICIPIO, REGIAO_SAUDE, MACRORREGIAO)
VALUES (-1, '-1', 'NAO INFORMADO', 'NAO INFORMADO', 'NAO INFORMADO', 'NAO INFORMADO',
        'NAO INFORMADO', 'NAO INFORMADO')
"""
executar(MEMBRO_DESCONHECIDO)

In [ ]:
SQL = """
MERGE INTO DIM_ESTABELECIMENTO d
USING (
    WITH atributos AS (
        SELECT COD_CNES,
               STATS_MODE(TIPO_UNIDADE)     AS TIPO_UNIDADE,
               STATS_MODE(TIPO_GESTAO)      AS TIPO_GESTAO,
               STATS_MODE(COD_IBGE_6_ATEND) AS COD_IBGE_6
        FROM SILVER_SIA
        GROUP BY COD_CNES
    ),
    cnes AS (
        SELECT COD_CNES FROM SILVER_ESTABELECIMENTO
        UNION SELECT COD_CNES FROM atributos
        UNION SELECT COD_CNES FROM SILVER_SIH
        UNION SELECT COD_CNES FROM SILVER_LEITOS
    )
    SELECT c.COD_CNES,
           NVL(e.NOME_ESTABELECIMENTO, 'NAO CADASTRADO - ' || c.COD_CNES) AS NOME_ESTABELECIMENTO,
           NVL(a.TIPO_UNIDADE, 'NAO INFORMADO')   AS TIPO_UNIDADE,
           NVL(a.TIPO_GESTAO, 'NAO INFORMADO')    AS TIPO_GESTAO,
           NVL(m.NOME_MUNICIPIO, 'NAO INFORMADO') AS NOME_MUNICIPIO,
           NVL(m.REGIAO_SAUDE, 'NAO INFORMADO')   AS REGIAO_SAUDE,
           NVL(m.MACRORREGIAO, 'NAO INFORMADO')   AS MACRORREGIAO
    FROM cnes c
    LEFT JOIN SILVER_ESTABELECIMENTO e ON e.COD_CNES = c.COD_CNES
    LEFT JOIN atributos a              ON a.COD_CNES = c.COD_CNES
    LEFT JOIN SILVER_MUNICIPIO m       ON m.COD_IBGE_6 = NVL(e.COD_IBGE_6, a.COD_IBGE_6)
    WHERE c.COD_CNES IS NOT NULL
) s
ON (d.COD_CNES = s.COD_CNES)
WHEN MATCHED THEN UPDATE SET
    d.NOME_ESTABELECIMENTO = s.NOME_ESTABELECIMENTO, d.TIPO_UNIDADE = s.TIPO_UNIDADE,
    d.TIPO_GESTAO = s.TIPO_GESTAO, d.NOME_MUNICIPIO = s.NOME_MUNICIPIO,
    d.REGIAO_SAUDE = s.REGIAO_SAUDE, d.MACRORREGIAO = s.MACRORREGIAO
  WHERE DECODE(d.NOME_ESTABELECIMENTO, s.NOME_ESTABELECIMENTO, 0, 1) + DECODE(d.TIPO_UNIDADE, s.TIPO_UNIDADE, 0, 1)
      + DECODE(d.TIPO_GESTAO, s.TIPO_GESTAO, 0, 1) + DECODE(d.NOME_MUNICIPIO, s.NOME_MUNICIPIO, 0, 1)
      + DECODE(d.REGIAO_SAUDE, s.REGIAO_SAUDE, 0, 1) + DECODE(d.MACRORREGIAO, s.MACRORREGIAO, 0, 1) > 0
WHEN NOT MATCHED THEN INSERT
    (SK_ESTABELECIMENTO, COD_CNES, NOME_ESTABELECIMENTO, TIPO_UNIDADE, TIPO_GESTAO,
     NOME_MUNICIPIO, REGIAO_SAUDE, MACRORREGIAO)
VALUES
    (SEQ_DIM_ESTABELECIMENTO.NEXTVAL, s.COD_CNES, s.NOME_ESTABELECIMENTO, s.TIPO_UNIDADE,
     s.TIPO_GESTAO, s.NOME_MUNICIPIO, s.REGIAO_SAUDE, s.MACRORREGIAO)
"""

In [ ]:
mesclar("DIM_ESTABELECIMENTO", SQL, origem="SILVER_ESTABELECIMENTO")

In [ ]:
consultar("SELECT * FROM DIM_ESTABELECIMENTO ORDER BY 1 FETCH FIRST 10 ROWS ONLY")